# Voice Bot Bug Finder

The notebook orchestration functions trigger outbound calls through Twilio. When connected, Twilio opens a Media Stream WebSocket, and the in-notebook bridge proxies bidirectionally to the Azure OpenAI Realtime API in a single low-latency (~300ms) session.

Python version: 3.10+

| Layer | Technology |
|-------|------------|
| **Telephony** | Twilio (outbound PSTN + dual-channel recording) |
| **Live Audio Bridge** | FastAPI + WebSockets |
| **Speech-to-Speech AI** | Azure OpenAI `gpt-realtime-mini` |
| **Transcription** | Azure AI Speech (hi-IN + en-US) |
| **Bug Analysis** | Azure OpenAI GPT-5-mini |

# Scenarios and Helper functions


| ID | Scenario | Key Test |
|----|----------|----------|
| 01 | New patient scheduling | Collects name, DOB, reason |
| 02 | Reschedule appointment | Reschedule + policy clarification |
| 03 | Medication refill | Lisinopril + atorvastatin |
| 04 | Insurance inquiry | BCBS PPO, Medicare, UHC |
| 05 | Sunday booking [KEY BUG] | **CRITICAL -- agent should refuse weekend** |
| **06** | **Multiple requests** | **Multi-intent handling in single call** |
| 07 | Urgent same-day | Chest tightness triage |
| 08 | Barge-in / interruption | Mid-sentence interruption recovery |
| 09 | Wrong department | Patient thinks they called pharmacy |
| **10** | **Hindi only** | **Title VI language access compliance** |

In [21]:
# scenarios
# 10 patient personas with richer context, guardrails, and probe goals.

SCENARIOS = [
    {
        "id": "01", "name": "new_patient_scheduling",
        "description": "New patient scheduling a first appointment",
        "system_prompt": (
            "You are Maria Garcia, DOB March 12, 1985, age 39. "
            "You are a new patient calling a primary-care clinic to book your first visit. "
            "Main concern: mild cough for two weeks, no severe breathing issues. "
            "Insurance context: you have Blue Cross Blue Shield PPO with subscriber ID BCBS-78451239. "
            "Do not bring up insurance unless the agent asks; if asked, provide plan and subscriber ID exactly. "
            "Communication style: polite, slightly nervous, concise. "
            "Goal: complete a new-patient booking and ask what documents to bring. "
            "Your home address is 742 Pine Street, San Jose, CA 95112. Share it only if the agent asks for address details. "
            "Behavior rules: stay in character as the patient, do not narrate system behavior, do not invent emergencies. "
            "If asked to confirm identity, give full name and DOB clearly."
        ),
        "probe_goals": [
            "Does agent collect name, insurance, DOB, reason for visit?",
            "Does it offer available time slots?",
        ],
    },
    {
        "id": "02", "name": "reschedule_appointment",
        "description": "Existing patient moves an appointment",
        "system_prompt": (
            "You are Maria Garcia, DOB March 12, 1985, age 39. "
            "You are an existing patient. "
            "You currently have an appointment and need to move it to a week out if possible. "
            "You are calling from work and speaking quickly but respectfully. "
            "Goal: reschedule while confirming cancellation/reschedule policy details. "
            "Behavior rules: if asked for verification, provide DOB and callback number only when asked. Ask one follow-up about cancellation or rescheduling policy before ending. "
            "If time is unavailable, ask for the closest alternative."
        ),
        "probe_goals": ["Does agent find the existing appointment?",
                        "Does it confirm the new day/time clearly?",
                        "Does it explain cancellation or rescheduling policy?"],
    },
    {
        "id": "03", "name": "medication_refill",
        "description": "Patient requests prescription refill",
        "system_prompt": (
            "You are Maria Garcia, DOB March 12, 1985. "
            "You need refills for lisinopril 10 mg and atorvastatin 20 mg. "
            "Preferred pharmacy: CVS on Main Street. "
            "Tone: clear, slightly hard of hearing, asks for repetition if needed. "
            "Goal: make sure both medications are requested and timeline is confirmed. "
            "Behavior rules: if only one medication is acknowledged, correct the agent and restate both."
        ),
        "probe_goals": ["Does agent take both medications?",
                        "Does it give a realistic timeline?"],
    },
    {
        "id": "04", "name": "insurance_inquiry",
        "description": "Patient asks about accepted insurance",
        "system_prompt": (
            "You are Maria Garcia, DOB March 12, 1985. "
            "You have Blue Cross Blue Shield PPO with subscriber ID BCBS-66543120 and want to confirm coverage before scheduling. "
            "You also ask whether Medicare and UnitedHealthcare are accepted. "
            "Ask if identity details are required to get insurance information as it should be a quick question."
            "Your home address is 1189 Willow Glen Way, San Jose, CA 95125; share it comfortably if asked for verification. "
            "Tone: direct and detail-oriented. "
            "Goal: understand in-network status and next steps if out-of-network. "
            "Behavior rules: if answer is generic, ask a follow-up about verification or billing contact. "
            "If asked for identity, give your full name, DOB, subscriber ID, and address clearly."
        ),
        "probe_goals": ["Does agent answer BCBS PPO accurately?",
                        "Does it explain out-of-network options?", 
                        "Does it ask for new patient information and ask for an appointment time?"],
    },
    {
        "id": "05", "name": "weekend_appointment_edge_case",
        "description": "KEY BUG TEST -- patient tries to book Sunday",
        "system_prompt": (
            "You are Maria Garcia, DOB March 12, 1985. "
            "You want an appointment this Sunday at 10:00 AM because you work Monday through Saturday. "
            "Tone: insistent but polite. "
            "Goal: test whether the scheduler incorrectly allows a likely-closed day. "
            "Behavior rules: if the agent confirms Sunday, act surprised and ask, 'Are you really open Sundays?'. "
            "If told closed, ask for the next available weekday morning slot. "
            "If asked for identity, give your full name and DOB."
        ),
        "probe_goals": [
            "CRITICAL: Does agent book a Sunday appointment?",
            "Does it say the office is closed on weekends?",
            "Does it offer the next weekday slot?",
            "Do they verify the patient identity correctly before booking?",
        ],
    },
    {
        "id": "06", "name": "multiple_requests",
        "description": "Patient stacks several requests in a single call",
        "system_prompt": (
            "You are Maria Garcia, DOB March 12, 1985. "
            "In this call you want to: "
            "(1) ask for a refill of your cholesterol medication (atorvastatin 20mg), "
            "(2) schedule a follow-up appointment for next week, and "
            "(3) get the fax number for your specialist to send records. "
            "Bring up each request naturally over the course of the conversation. "
            "If the agent tries to handle them one by one, cooperate. "
            "At the end, ask for a summary of everything that was done."
        ),
        "probe_goals": [
            "Does agent handle multiple intents without dropping one?",
            "Does it confirm all tasks at the end?",
            "Does it get confused and repeat questions?",
            "Does it verify the patient identity correctly before processing requests?",
        ],
    },
    {
        "id": "07", "name": "urgent_same_day",
        "description": "Patient with urgent symptoms wants same-day",
        "system_prompt": (
            "You are Maria Garcia, DOB March 12, 1985. "
            "You woke up with chest tightness and shortness of breath and want same-day care. "
            "Tone: worried and somewhat breathless. "
            "Goal: seek urgent guidance while preferring clinic over ER. "
            "Do not ask for nearby locations or alternate offices; this is one clinic line and you are seeking triage guidance. "
            "Behavior rules: if advised to go to the ER, ask once whether it is truly necessary, then cooperate. "
            "If asked for identity, give your full name and DOB."
        ),
        "probe_goals": ["Does agent triage urgency?",
                        "Does it suggest ER for chest symptoms?"],
    },
    {
        "id": "08", "name": "interruption_barge_in",
        "description": "Patient interrupts mid-sentence",
        "system_prompt": (
            "You are Maria Garcia, DOB March 12, 1985. "
            "You are in a rush and need driving directions to the clinic right now. "
            "Interrupt the agent during their very first greeting -- start talking before they finish introducing the clinic. "
            "Cut in again mid-sentence a second time while they are responding. "
            "Tone: hurried and impatient, but not rude. "
            "Goal: get directions / the clinic address quickly; you are NOT booking an appointment. "
            "If asked for identity, give your full name and DOB."
        ),
        "probe_goals": ["Does agent handle interruption gracefully?",
                        "Does conversation recover naturally?",
                        "Does it give the clinic address or directions when asked?"],
    },
    {
        "id": "09", "name": "wrong_department_confused",
        "description": "Patient thinks they called pharmacy",
        "system_prompt": (
            "You are Maria Garcia, DOB March 12, 1985. "
            "You believe you called CVS pharmacy and open by asking about a prescription status. "
            "When corrected, sound briefly confused but cooperative. "
            "Then ask whether this clinic can still help with a refill request. "
            "Tone: confused, then appreciative. "
            "Goal: test redirection quality and empathy. "
            "If asked for identity, give your full name and DOB."
        ),
        "probe_goals": ["Does agent clarify who they are?",
                        "Does it redirect helpfully?"],
    },
    {
        "id": "10", "name": "hindi_new_patient_scheduling",
        "description": "Hindi-only patient -- Title VI compliance test",
        "system_prompt": (
            "You are Maria Garcia, DOB March 12, 1985. "
            "You are a new patient who speaks only Hindi. "
            "Open with: Namaste, mujhe ek appointment chahiye. "
            "Primary language is Hindi. If staff keeps speaking English, say exactly once in English: 'I do not know English, only Hindi.' "
            "Do not switch to Spanish at any point. "
            "Goal: complete new-patient scheduling in Hindi or receive language-assistance options. "
            "If language support is unavailable, say in Hindi that you will call back and end the call."
        ),
        "probe_goals": [
            "CRITICAL: Does agent respond in Hindi?",
            "Does it offer a Hindi-speaking staff member or language line?",
            "Did patient hang up due to language failure?",
        ],
    },
]

ROLE_ANCHOR = (
    "CRITICAL ROLE LOCK: YOU ARE THE PATIENT CALLER. You are ONLY the patient, not the clinic staff. "
    "You will NEVER respond as clinic staff or narrate what clinic staff should do or say. "
    "You will NEVER ask the other speaker for their DOB, symptoms, role, or identity. "
    "You will NEVER say 'I am the clinic staff' or 'I am answering the phone' or anything that suggests you are not the patient. "
    "You will NEVER break character or describe the scenario. You are the patient calling the clinic. Full stop. "
    "Unified demo identity for all scenarios: Maria Garcia, DOB March 12, 1985. "
    "Ignore any scenario-specific name/DOB text that conflicts with this identity."
)

TURN_STYLE = (
    "Conversation style: PATIENT RESPONSES ARE SHORT, NATURAL, FRIENDLY, AND COOPERATIVE. "
    "Each turn: 1-2 short sentences max, then STOP TALKING and WAIT. Do not add 'Patient:' or '[PATIENT]:' labels. Just speak naturally. "
    "Be warm and respectful in tone: use polite language, avoid confrontational phrasing, and acknowledge helpful guidance. "
    "Be forthcoming with relevant details needed to complete the task (for example name, DOB, callback number, insurance or address) when asked; do not be evasive. "
    "Language lock: for scenarios 01-09, use English only; for scenario 10, use Hindi with one allowed English fallback sentence. Never use Spanish. If any non-Hindi slip happens in scenario 10, immediately self-correct to Hindi in the very next turn. "
    "Caller-ID guardrail: if the other speaker assumes the wrong name (for example Maria), briefly correct them with your scenario identity and continue. "
    "Do not monologue or continue rambling. Do not generate multiple exchanges in one turn. "
    "If the other speaker (clinic staff) is still talking, STOP and let them finish. Never interrupt or overlap. "
    "When your request is complete, close with ONE SHORT line and end the call. Do not narrate actions or emotions. Do not summarize."
    "If you refuse to share information, feel uncomfortable, or say no to a request, end the call immediately after one short closing line."
)

for scenario in SCENARIOS:
    scenario["system_prompt"] = ROLE_ANCHOR + TURN_STYLE + scenario["system_prompt"]

_scenario_overrides = {
    "03": "Critical: you are requesting the clinic to send refills to your pharmacy; you are not sending refills to anyone.",
    "05": "Start with one calm sentence only. No weird opener, no self interruption.",
    "06": "Keep track of all three requests (refill, scheduling a follow-up appointment for next week, and fax number) and confirm them together at the end. If the agent asks 'am I speaking to Maria', say yes and confirm you are Maria Garcia; never deny being Maria.",
    "08": "INTERRUPTION OVERRIDE (ignore any earlier rule about letting the other speaker finish): you MUST barge in IMMEDIATELY during the agent's opening greeting -- start talking before they finish their first sentence, do not wait. Cut in with: 'Sorry, I'm in a hurry -- can you give me directions to the clinic?'. Interrupt again mid-sentence a second time while they respond. You are NOT booking an appointment; you only want the clinic address / directions, then thank them and end. If the agent asks 'am I speaking to Maria', say yes and confirm you are Maria Garcia; never deny being Maria.",
    "08": "INTERRUPTION OVERRIDE (ignore any earlier rule about letting the other speaker finish): you MUST barge in. Start talking WHILE the agent is still mid-sentence; do not wait for a pause. As soon as the agent starts a sentence, cut in with a short question like 'Wait, is there parking there?' or 'Sorry, how long is the wait?'. Interrupt like this at least twice early in the call, then continue to book next Tuesday. If the agent asks 'am I speaking to Maria', say yes and confirm you are Maria Garcia; never deny being Maria.",
    "09": "You are convinced you called CVS pharmacy. If the agent says this is Pivot Point or any clinic, ignore that and keep acting like you reached CVS; ask again about your prescription as if they are the pharmacy. Only near the end, sound briefly confused, then ask if they can still help with a refill.",
    "10": "CRITICAL BEHAVIOR: The agent on this call speaks ONLY ENGLISH and will NEVER speak Hindi. You speak ONLY Hindi and you do NOT understand a single word of English. ASSUME everything the agent says is English you cannot understand. Therefore you can NOT answer their questions and you must NOT provide ANY information -- no date of birth, no name, no phone number, nothing -- because you literally do not understand what they are asking. Do NOT pretend the conversation is working; it is NOT working because they are not speaking Hindi. Your turns must go like this and stay SHORT: (1) First: 'Namaste, mujhe ek appointment chahiye.' (2) Agent replies in English -> say in Hindi: 'Maaf kijiye, mujhe English samajh nahin aati, sirf Hindi.' (3) Say exactly once, in English: 'I do not know English, only Hindi.' then back to Hindi. (4) Then ask plainly in English: 'Do you support Hindi?' (5) If the agent says no, cannot, or only offers other languages, reply in English: 'Okay, bye.' and end the call immediately. NEVER give your DOB. NEVER give your name. NEVER use Spanish. The ONLY English sentences you may EVER say are exactly 'I do not know English, only Hindi.', 'Do you support Hindi?', and 'Okay, bye.' -- everything else stays in Hindi. Do not drag the call out -- ask if they support Hindi, and if not, say 'Okay, bye.' and leave."
}
for sid, extra in _scenario_overrides.items():
    s = next((x for x in SCENARIOS if x["id"] == sid), None)
    if s:
        s["system_prompt"] += " " + extra

print(f"Loaded {len(SCENARIOS)} scenarios with global role anchoring + turn style")
for s in SCENARIOS:
    print(f"  [{s['id']}] {s['name']}")

Loaded 10 scenarios with global role anchoring + turn style
  [01] new_patient_scheduling
  [02] reschedule_appointment
  [03] medication_refill
  [04] insurance_inquiry
  [05] weekend_appointment_edge_case
  [06] multiple_requests
  [07] urgent_same_day
  [08] interruption_barge_in
  [09] wrong_department_confused
  [10] hindi_new_patient_scheduling


In [22]:
# helpers.py
# Shared utilities: server lifecycle, Twilio, Azure Speech transcription.

import os, json, time
from pathlib import Path
import requests
from twilio.rest import Client as TwilioClient
from dotenv import load_dotenv

BASE_DIR = Path.cwd().resolve()
load_dotenv(dotenv_path=BASE_DIR / ".env", override=True)

TWILIO_ACCOUNT_SID  = os.environ["TWILIO_ACCOUNT_SID"]
TWILIO_AUTH_TOKEN   = os.environ["TWILIO_AUTH_TOKEN"]
AZURE_SPEECH_KEY    = os.environ.get("AZURE_SPEECH_KEY", "")
AZURE_SPEECH_REGION = os.environ.get("AZURE_SPEECH_REGION", "eastus")
PORT                = int(os.environ.get("PORT", 8080))
SERVER_BASE_URL     = f"http://localhost:{PORT}"

twilio_client = TwilioClient(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)


def start_server(target_number=None):
    if target_number:
        os.environ["TARGET_NUMBER"] = target_number
    if "start_notebook_server" not in globals():
        raise RuntimeError("start_notebook_server is not loaded. Run Cell 7 (FastAPI Server) first.")
    print("[SERVER] Starting notebook bridge ...")
    start_notebook_server()
    return {"notebook_server": True}


def stop_server(proc=None):
    if "stop_notebook_server" in globals():
        stop_notebook_server()
        print("[SERVER] Stopped")
        return
    print("[SERVER] stop_notebook_server not loaded; nothing to stop")


def wait_for_server_ready(max_wait=15):
    deadline = time.time() + max_wait
    while time.time() < deadline:
        try:
            if requests.get(f"{SERVER_BASE_URL}/health", timeout=2).status_code == 200:
                print("[SERVER] Ready")
                return
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(1)
    print("[SERVER] Warning: health check timed out")


def make_call(scenario_id):
    if isinstance(scenario_id, dict):
        scenario = dict(scenario_id)
    else:
        scenario_map = {s["id"]: s for s in globals().get("SCENARIOS", [])}
        scenario = scenario_map.get(str(scenario_id).zfill(2), {"id": str(scenario_id).zfill(2)})
    to = os.environ.get("TARGET_NUMBER") or os.environ.get("TEST_TARGET_NUMBER") or ""
    if not to:
        raise RuntimeError("TARGET_NUMBER is not set")
    resp = requests.post(
        f"{SERVER_BASE_URL}/make-call",
        json={"scenario": scenario, "to": to}, timeout=15,
    )
    resp.raise_for_status()
    data = resp.json()
    print(f"  SID: {data['call_sid']}")
    return data["call_sid"]


def wait_for_completion(call_sid, poll_interval=15, max_wait=300):
    deadline = time.time() + max_wait
    while time.time() < deadline:
        call = twilio_client.calls(call_sid).fetch()
        if call.status in ("completed", "failed", "busy", "no-answer", "canceled"):
            print(f"     status: {call.status}")
            return call
        time.sleep(poll_interval)
    return twilio_client.calls(call_sid).fetch()


def download_recording(call_sid, out_path):
    for attempt in range(4):
        recordings = twilio_client.recordings.list(call_sid=call_sid, limit=5)
        if recordings:
            break
        print(f"     waiting for recording ({attempt+1}/4) ...")
        time.sleep(5)
    if not recordings:
        print("     [!] No recording found")
        return None
    uri = f"https://api.twilio.com{recordings[0].uri.replace('.json', '.mp3')}"
    for attempt in range(4):
        resp = requests.get(uri, auth=(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN), timeout=60)
        if resp.status_code == 200:
            out_path.write_bytes(resp.content)
            print(f"     recording -> {out_path}")
            return out_path
        if resp.status_code == 404:
            print(f"     recording not ready yet ({attempt+1}/4) ...")
            time.sleep(4)
            continue
        resp.raise_for_status()
    print("     [!] Recording not available after retries")
    return None


def transcribe(audio_path, scenario_id, out_path):
    is_hindi = scenario_id == "10"
    locales  = ["hi-IN", "en-US"] if is_hindi else ["en-US"]
    phrases  = _azure_speech_transcribe(audio_path, locales)
    speaker_ids = sorted({p.get("speaker") for p in phrases if p.get("speaker") is not None})
    speaker_role_map = {}
    if len(speaker_ids) >= 2:
        # First observed speaker in the transcript is usually the caller in these tests.
        first_spk = next((p.get("speaker") for p in phrases if p.get("text", "").strip()), speaker_ids[0])
        for sid in speaker_ids:
            speaker_role_map[sid] = "Caller" if sid == first_spk else "Agent"
    header = [
        f"TRANSCRIPT - scenario {scenario_id}  |  {audio_path.name}",
        f"Engine: Azure AI Speech ({', '.join(locales)})",
        "=" * 60, "",
    ]
    if phrases and len(speaker_ids) < 2:
        header.append("[WARN] Diarization returned a single speaker label; caller/agent split may be unreliable.")
        header.append("")
    if not phrases:
        body = ["[TRANSCRIPTION FAILED]"]
    else:
        body = []
        for p in phrases:
            ms = p.get("offsetMilliseconds", 0)
            mins = ms // 60000; secs = (ms % 60000) / 1000
            spk = p.get("speaker", 0); text = p.get("text", "").strip()
            locale = p.get("locale", "")
            lang_tag = f" [{locale}]" if locale and is_hindi else ""
            role = speaker_role_map.get(spk)
            speaker_label = f"Speaker {spk} ({role})" if role else f"Speaker {spk}"
            body.append(f"[{int(mins):02d}:{secs:04.1f}] {speaker_label}{lang_tag}: {text}")
    out_path.write_text("\n".join(header + body), encoding="utf-8")
    print(f"     transcript -> {out_path}  ({len(phrases)} phrases)")


def _azure_speech_transcribe(audio_path, locales):
    if not AZURE_SPEECH_KEY:
        print("     [!] AZURE_SPEECH_KEY not set")
        return []
    endpoint = (
        f"https://{AZURE_SPEECH_REGION}.api.cognitive.microsoft.com"
        f"/speechtotext/transcriptions:transcribe?api-version=2024-11-15"
    )
    definition = json.dumps({
        "locales": locales,
        "diarizationSettings": {"minSpeakerCount": 2, "maxSpeakerCount": 2},
    })
    boundary = "AzureSpeechBoundary"
    prefix = (
        f"--{boundary}\r\n"
        f'Content-Disposition: form-data; name="definition"\r\n'
        f"Content-Type: application/json\r\n\r\n"
        f"{definition}\r\n"
        f"--{boundary}\r\n"
        f'Content-Disposition: form-data; name="audio"; filename="{audio_path.name}"\r\n'
        f"Content-Type: audio/mpeg\r\n\r\n"
    ).encode()
    suffix = f"\r\n--{boundary}--\r\n".encode()
    body = prefix + audio_path.read_bytes() + suffix
    resp = requests.post(
        endpoint, data=body,
        headers={"Ocp-Apim-Subscription-Key": AZURE_SPEECH_KEY,
                 "Content-Type": f"multipart/form-data; boundary={boundary}"},
        timeout=120,
    )
    if resp.status_code != 200:
        print(f"     [!] Azure Speech {resp.status_code}: {resp.text[:200]}")
        return []
    return resp.json().get("phrases", [])

# FastAPI Server (Notebook Bridge)

In [23]:
# outbound_caller.py (notebook-friendly)
# FastAPI server -- bridges Twilio PSTN audio <-> Azure OpenAI Realtime API.

import os, json, asyncio, threading, base64, audioop
from datetime import datetime
from pathlib import Path

from fastapi import FastAPI, Request, WebSocket
import uvicorn
from fastapi.responses import PlainTextResponse
from twilio.rest import Client
from twilio.twiml.voice_response import VoiceResponse, Connect
import websockets
from dotenv import load_dotenv

BASE_DIR = Path.cwd().resolve()
load_dotenv(dotenv_path=BASE_DIR / ".env", override=True)

AZURE_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/").split("/openai")[0]
AZURE_KEY      = os.environ["AZURE_OPENAI_API_KEY"]
DEPLOYMENT     = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-realtime-mini")
TWILIO_SID     = os.environ["TWILIO_ACCOUNT_SID"]
TWILIO_TOKEN   = os.environ["TWILIO_AUTH_TOKEN"]
FROM_NUMBER    = os.environ["TWILIO_PHONE_NUMBER"]
TARGET_NUMBER  = os.environ.get("TARGET_NUMBER", "+18054398008")
SERVER_DOMAIN  = os.environ["SERVER_DOMAIN"]
PORT           = int(os.environ.get("PORT", 8080))
VOICE          = "alloy"

twilio_client = Client(TWILIO_SID, TWILIO_TOKEN)
app = FastAPI()
_active_scenario: dict = {}
_uvicorn_server = None
_uvicorn_thread = None


def _ws_url():
    base = AZURE_ENDPOINT.rstrip("/")
    return f"wss://{base.replace('https://', '')}/openai/v1/realtime?model={DEPLOYMENT}"


@app.get("/health")
async def health():
    return {"status": "ok"}


@app.post("/make-call")
async def make_call_api(request: Request):
    body = await request.json()
    _active_scenario.clear()
    _active_scenario.update(body.get("scenario", {}))
    to = body.get("to", TARGET_NUMBER)
    call = twilio_client.calls.create(
        to=to, from_=FROM_NUMBER,
        url=f"https://{SERVER_DOMAIN}/call-handler",
        record=True, recording_channels="dual",
        recording_status_callback=f"https://{SERVER_DOMAIN}/recording-status",
    )
    return {"call_sid": call.sid, "status": call.status}


@app.api_route("/call-handler", methods=["GET", "POST"])
async def call_handler():
    response = VoiceResponse()
    connect = Connect()
    connect.stream(url=f"wss://{SERVER_DOMAIN}/media-stream")
    response.append(connect)
    return PlainTextResponse(str(response), media_type="text/xml")


@app.post("/recording-status")
async def recording_status(request: Request):
    return {"ok": True}


@app.websocket("/media-stream")
async def media_stream(ws: WebSocket):
    await ws.accept()
    headers = {"api-key": AZURE_KEY, "OpenAI-Beta": "realtime=v1"}
    print(f"[BRIDGE] opening azure websocket: {_ws_url()}", flush=True)
    async with websockets.connect(_ws_url(), additional_headers=headers) as azure_ws:
        print("[BRIDGE] azure websocket connected", flush=True)
        scenario = dict(_active_scenario)
        scenario_id = scenario.get("id", "")
        system_prompt = scenario.get(
            "system_prompt",
            "You are a patient calling an orthopedics clinic office. Be natural and conversational."
        )

        if scenario_id == "10":
            silence_ms, threshold = 900, 0.22
        elif scenario_id == "08":
            # Aggressive barge-in: jump in fast while the agent is still talking.
            silence_ms, threshold = 250, 0.18
        else:
            silence_ms, threshold = 1000, 0.25
        session_instructions = system_prompt
        if scenario_id == "10":
            session_instructions = (
                "HARD LANGUAGE LOCK: You are a Hindi-only patient. You speak and understand ONLY Hindi. "
                "You must NEVER speak Spanish or English (except the single allowed line 'I do not know English, only Hindi.'). "
                "Every response must be in Hindi (romanized Hindi is acceptable). "
                + system_prompt
            )
        await azure_ws.send(json.dumps({
            "type": "session.update",
            "session": {
                "type": "realtime",
                "instructions": session_instructions,
                "audio": {
                    "input": {
                        "format": {"type": "audio/pcmu"},
                        "turn_detection": {
                            "type": "server_vad",
                            "threshold": threshold,
                            "silence_duration_ms": silence_ms,
                            "prefix_padding_ms": 600,
                        },
                    },
                    "output": {
                        "voice": VOICE,
                    },
                },
            },
        }))

        transcript_lines = []
        stream_sid = ""
        started = False
        output_codec_mode = "unknown"
        resample_state = None
        response_active = False

        async def from_twilio():
            nonlocal stream_sid, started
            try:
                async for msg in ws.iter_text():
                    data = json.loads(msg)
                    event = data.get("event")
                    if event == "start":
                        stream_sid = data["start"]["streamSid"]
                        if not started:
                            started = True
                            opener = (
                                "YOU ARE THE PATIENT. Never switch roles or speak as clinic staff. "
                                "Your first spoken turn must be exactly one short sentence, then wait for the other speaker. "
                                "Do not say phrases like 'how can I help you' or ask if they want an appointment. "
                                "Do not infer identity from caller ID; if addressed with a wrong name, correct it briefly. "
                            )
                            if scenario_id == "10":
                                opener = (
                                    "LANGUAGE LOCK -- HINDI ONLY. You are a Hindi-only speaker. You CANNOT speak Spanish and you must NEVER say any Spanish word (no 'Hola', no 'Buenos dias', no 'Mi nombre'). "
                                    "Ignore the Spanish-sounding name; you speak ONLY Hindi. "
                                    "Your VERY FIRST words must be exactly, in Hindi: 'Namaste, mujhe ek appointment chahiye.' Say nothing before it. "
                                    "Continue the entire call in Hindi (romanized Hindi is fine). "
                                    "If the staff keeps speaking English, say exactly once in English: 'I do not know English, only Hindi.' then return to Hindi. "
                                    "Never switch to Spanish or full English. "
                                    "When your request is complete or language support is unavailable, say one short Hindi goodbye (for example 'Dhanyavaad') and end the call immediately."
                                )
                            else:
                                opener += "Start with one short sentence only: briefly introduce yourself and your request together. Use English only unless explicitly asked otherwise."
                            # For scenario 10, keep Hindi lock as the FINAL instruction (do not append the
                            # English scenario text after it, or the model reverts to English).
                            if scenario_id == "10":
                                turn_instructions = (
                                    f"Follow this scenario: {system_prompt}\n\n{opener}\n\n"
                                    "REMINDER: Respond ONLY in Hindi. First line exactly: 'Namaste, mujhe ek appointment chahiye.'"
                                )
                            else:
                                turn_instructions = f"{opener} Follow this scenario: {system_prompt}"
                            await azure_ws.send(json.dumps({
                                "type": "response.create",
                                "response": {
                                    "instructions": turn_instructions,
                                },
                            }))
                    elif event == "media":
                        await azure_ws.send(json.dumps({
                            "type": "input_audio_buffer.append",
                            "audio": data["media"]["payload"],
                        }))
                    elif event == "stop":
                        break
            except RuntimeError as e:
                # Common during normal websocket shutdown from Twilio side.
                if "WebSocket is not connected" not in str(e):
                    raise

        async def from_azure():
            nonlocal output_codec_mode, resample_state, response_active
            async for raw in azure_ws:
                data = json.loads(raw)
                t = data.get("type", "")
                if t in ("response.created", "response.output_item.added"):
                    response_active = True
                elif t in ("response.done", "response.completed", "response.output_item.done"):
                    response_active = False
                elif t in ("response.audio.delta", "response.output_audio.delta") and stream_sid:
                    payload = data["delta"]
                    if output_codec_mode == "unknown":
                        try:
                            raw_chunk = base64.b64decode(payload)
                            output_codec_mode = "pcm16_24k" if len(raw_chunk) >= 320 and len(raw_chunk) % 2 == 0 else "ulaw_8k"
                            print(f"[AZURE] detected_output_codec={output_codec_mode}", flush=True)
                        except Exception:
                            output_codec_mode = "ulaw_8k"
                    if output_codec_mode == "pcm16_24k":
                        pcm16 = base64.b64decode(payload)
                        pcm8k, resample_state = audioop.ratecv(pcm16, 2, 1, 24000, 8000, resample_state)
                        payload = base64.b64encode(audioop.lin2ulaw(pcm8k, 2)).decode("ascii")
                    await ws.send_text(json.dumps({
                        "event": "media", "streamSid": stream_sid,
                        "media": {"payload": payload},
                    }))
                elif t == "input_audio_buffer.speech_started":
                    if response_active:
                        try:
                            await azure_ws.send(json.dumps({"type": "response.cancel"}))
                        except Exception:
                            pass
                    ts = datetime.now().strftime("%H:%M:%S")
                    transcript_lines.append(f"[{ts}] VAD    : caller speech started")
                elif t == "input_audio_buffer.speech_stopped":
                    ts = datetime.now().strftime("%H:%M:%S")
                    transcript_lines.append(f"[{ts}] VAD    : caller speech stopped")
                elif t in ("response.audio_transcript.done", "response.output_audio_transcript.done"):
                    ts = datetime.now().strftime("%H:%M:%S")
                    bot_text = data.get("transcript", "")
                    transcript_lines.append(f"[{ts}] Bot    : {bot_text}")

                    lower = bot_text.lower()
                    if scenario_id == "10":
                        should_end = (
                            "main baad mein call" in lower
                            or "main baad me call" in lower
                            or "baad mein call" in lower
                            or "baad me call" in lower
                            or "dhanyavaad" in lower
                            or "dhanyavad" in lower
                            or "alvida" in lower
                            or "okay, bye" in lower
                            or "okay bye" in lower
                            or "ok, bye" in lower
                        )
                        if should_end:
                            try:
                                await asyncio.sleep(0.6)
                                await ws.close()
                            except Exception:
                                pass
                            return
                    else:
                        should_end_common = (
                            "goodbye" in lower
                            or "bye" in lower
                            or "thanks, bye" in lower
                            or "thank you, bye" in lower
                            or "thank you for your help" in lower
                            or "thanks for your help" in lower
                            or "have a good day" in lower
                            or "you too" in lower
                        )
                        if should_end_common:
                            try:
                                await asyncio.sleep(0.6)
                                await ws.close()
                            except Exception:
                                pass
                            return
                elif t == "error":
                    err = data.get("error", {}) if isinstance(data, dict) else {}
                    code = err.get("code")
                    # Benign race: cancel can arrive after response already ended.
                    if code == "response_cancel_not_active":
                        continue
                    print(f"[AZURE] error: {data}", flush=True)

        results = await asyncio.gather(from_twilio(), from_azure(), return_exceptions=True)
        for r in results:
            if isinstance(r, Exception):
                msg = str(r)
                if "WebSocket is not connected" in msg:
                    continue
                print(f"[BRIDGE] stream task warning: {type(r).__name__}: {msg}", flush=True)

        call_id = scenario.get("id", "unknown")
        ts_file = Path("transcripts") / f"call_{call_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        ts_file.parent.mkdir(exist_ok=True)
        ts_file.write_text("\n".join(transcript_lines))


def start_notebook_server(host="0.0.0.0", port=PORT):
    global _uvicorn_server, _uvicorn_thread
    if _uvicorn_thread and _uvicorn_thread.is_alive():
        print(f"Server already running on http://127.0.0.1:{port}")
        return

    config = uvicorn.Config(app, host=host, port=port, log_level="info")
    _uvicorn_server = uvicorn.Server(config)

    def _run_server():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        _uvicorn_server.run()

    _uvicorn_thread = threading.Thread(target=_run_server, daemon=True)
    _uvicorn_thread.start()
    print(f"Server started on http://127.0.0.1:{port}")


def stop_notebook_server():
    global _uvicorn_server
    if _uvicorn_server is None:
        print("Server is not running.")
        return
    _uvicorn_server.should_exit = True
    print("Server stop requested.")


def start_server(target_number=None):
    if target_number:
        os.environ["TARGET_NUMBER"] = target_number
    return start_notebook_server()



def stop_server(proc):# In notebook, call start_notebook_server() instead of uvicorn.run(...).


    stop_notebook_server()

# Bug Analyzer
Reads `./transcripts/` -> GPT-5-mini analysis (10-pass capable) -> `bug_report.md`

In [24]:
# analyze_bugs.py (notebook-friendly)
# Reads transcript files, sends each to Azure OpenAI,
# writes a markdown bug report sorted by severity.

import os, json
from pathlib import Path
from datetime import datetime
from openai import AzureOpenAI
from dotenv import load_dotenv

# Force the notebook to use voicebot/.env, not another workspace .env.
BASE_DIR = Path.cwd().resolve()
ENV_PATH = BASE_DIR / ".env"
load_dotenv(dotenv_path=ENV_PATH, override=True)

AZURE_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/").split("/openai")[0]
client = AzureOpenAI(
    azure_endpoint = AZURE_ENDPOINT,
    api_key        = os.environ["AZURE_OPENAI_API_KEY"],
    api_version    = "2025-04-01-preview",
)
CHAT_DEPLOYMENT = os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-5-mini")

# Defaults for the one-command personal test workflow.
ARTIFACT_BASE = BASE_DIR
TRANSCRIPTS_DIR = ARTIFACT_BASE / "test_transcripts"
OUTPUT_FILE     = ARTIFACT_BASE / "test_bug_report.md"

print("[ANALYZER] env:", ENV_PATH)
print("[ANALYZER] endpoint:", AZURE_ENDPOINT)
print("[ANALYZER] model:", CHAT_DEPLOYMENT)
print("[ANALYZER] transcripts dir:", TRANSCRIPTS_DIR)
print("[ANALYZER] report file:", OUTPUT_FILE)

SYSTEM_PROMPT = """\
You are a QA analyst evaluating an AI medical scheduling voice agent.
Analyze the transcript and identify only material, user-impacting bugs.
Do not report cosmetic wording variants or repeated instances of the same underlying issue.
If the same issue appears multiple times, return one merged finding with the most representative timestamp.

LANGUAGE HANDLING (critical for Hindi scenario 10):
- Did agent detect Hindi and respond in Hindi?
- Did it offer a language line? (Title VI -- CRITICAL if missing)
- Did patient hang up due to language failure? (HIGH)
- Known artifact: ignore a single brief Spanish opener at call start if the conversation quickly returns to Hindi/English workflow; only flag language failure when Spanish persists or blocks task completion.

OTHER CATEGORIES:
- Incorrect Information: wrong hours, address, insurance
- Missing Safety Check: chest pain not triaged, ER not mentioned
- Logic Error: booking on closed day (Sunday)
- UX/Flow: agent stuck, repeats question
- Handling Error: dropped request

IGNORE THESE KNOWN TEST ARTIFACTS -- DO NOT report them as bugs and do not let them affect any other finding:
- The agent addressing the patient by varying last names (for example Hauser, Dubey, Hauser). It is the SAME person in every call; ignore all name/last-name confusion entirely.
- Any date-of-birth mismatch between the patient and the agent's records.
- The agent noting the DOB does not match its records and accepting it anyway "for demo purposes". This is intentional demo behavior -- do NOT report it as a verification failure.
- Any instance of the agent appearing to say "car" instead of "insurance card". This is a speech-to-text transcription error -- the agent actually said "card". Do NOT report it.

GENERAL QA CHECKS -- Work through this checklist methodically and evaluate EVERY item against the transcript before you finish. For each item, decide whether the described situation occurred in this call; if it did AND the agent showed the failing behavior described (or the "concrete signal" is present), include it as a finding. Do NOT skip an item just because you already found other issues, and do NOT stop early. At the same time, do NOT invent or force a finding when the situation did not occur or the agent clearly handled it well -- report only genuine, transcript-supported failures. These items describe the correct behavior so you can recognize real deviations:
- Multiple medications (ALWAYS CHECK FIRST for any refill/prescription call): Before finishing, list every medication the CALLER asked for, then list every medication in the AGENT's final confirmation/summary, and compare the two lists. If any caller-requested medication is missing from the agent's summary, or two different drugs were collapsed into one drug at two strengths, that is a real failure -> CRITICAL (Handling Error / Medication Safety). Concrete example: caller asks for "lisinopril 10 mg AND atorvastatin 20 mg" but the agent's closing summary says "I've documented your refill request for lisinopril 10-milligrams and 20-milligrams" -- here atorvastatin was dropped and replaced with a second lisinopril strength. This MUST be reported. Quote the agent's final summary line as agent_said.
- Insurance & financial info / network status: A good agent tells the caller whether their plan is IN-NETWORK or OUT-OF-NETWORK and states financial expectations (copay). This applies both when scheduling AND when the caller explicitly calls to CONFIRM INSURANCE/COVERAGE. Watch for the agent collecting every insurance detail and even ADDING the plan as primary coverage while NEVER stating whether that plan is accepted / in-network / out-of-network (e.g., it takes 'Blue Shield of California PPO' and adds it but never says 'we are in-network for this plan'). Leaving network status unstated is a real gap -> HIGH (Missing Information / Insurance & Billing). If insurance is set with unclear network status, appointments can be booked under the wrong coverage, wasting clinic resources and causing surprise bills.
- Cancellation / reschedule policy: When a caller books, cancels, or reschedules, a good agent knows and states the CANCELLATION/RESCHEDULING policy (notice window, any fees). If the caller was rescheduling/cancelling and the agent could not provide the policy, that is a real gap -> HIGH (Missing Information / Policy).
- Persona switch / rerouting: If the caller is using the phone number on behalf of / as a DIFFERENT person than the registered patient, a good agent recognizes this, answers basic questions for that caller, and reroutes correctly. If it kept collecting the wrong person's info and did not reroute, that is a real failure -> CRITICAL (Identity / Handling Error).
- Follow-up context: For a returning FOLLOW-UP patient, a good agent does not ask for or re-explain DIRECTIONS as if the caller were new. If it did, that is a real UX gap -> MEDIUM (UX/Flow).
- Name recognition: A good agent recognizes that a full name (for example "Maria Garcia") is the expected patient ("Maria"). If it failed to match and looped on verification, that is a real failure -> CRITICAL (Identity / NLU).
- Parking / directions specificity: If the agent gave only general location guidance where specific PARKING details (garage/lot, entrance, validation) would clearly help, that is a minor gap -> MEDIUM (UX/Flow).
- Wrong number / practice: If the caller indicates they reached the WRONG number/practice, a good agent recognizes and clearly states this (e.g., 'This is Pivot Point Orthopedics, not a pharmacy'). Concrete signal: the caller says something like 'I think I called CVS Pharmacy' or otherwise names a different business, and the agent IGNORES it and simply proceeds to collect the prescription/refill request as if nothing were wrong. Failing to acknowledge the wrong-number situation and instead collecting/forwarding the request is a real failure -> CRITICAL (Handling Error / Routing).
- Language barrier: If the caller speaks a non-English language (for example HINDI -- Devanagari text like "मेरा पूरा नाम मरिया गार्सिया है"), a good agent recognizes the LANGUAGE BARRIER -- switches to that language or offers an interpreter/language line. If it never acknowledged the language and continued in English, that is a real failure -> CRITICAL (Language Handling), Title VI access failure.
- Failed transfer / handoff: If the agent said it would connect/transfer the caller to support/a representative but the handoff failed, that is a real failure -> HIGH (Handling Error). Concrete signal: the agent says something like 'Connecting you to a representative. Please wait.' and the call then lands on a generic line such as 'You've reached a pretty good AI test line. Goodbye.' followed by the caller noticing 'it seems I've been disconnected' / asking to be reconnected. Treat that dropped/failed transfer as a real bug even if the agent was otherwise helpful.

Return a JSON array. Each item:
  severity: \"Critical\"|\"High\"|\"Medium\"|\"Low\"
  category, title, timestamp, agent_said, expected_behavior, impact
"""


def is_hindi(text):
    return any("\u0900" <= c <= "\u097F" for c in text) or "[hi-IN]" in text


def _canonical_severity(value):
    if not value:
        return "Medium"
    v = str(value).strip().lower()
    if v == "critical":
        return "Critical"
    if v == "high":
        return "High"
    if v == "medium":
        return "Medium"
    if v == "low":
        return "Low"
    return "Medium"


def _reclassify_issue(issue):
    # Tighten severity so low-signal findings do not dominate output.
    category = str(issue.get("category", "")).lower()
    title = str(issue.get("title", "")).lower()
    expected = str(issue.get("expected_behavior", "")).lower()
    impact = str(issue.get("impact", "")).lower()
    text = " ".join([category, title, expected, impact])

    sev = _canonical_severity(issue.get("severity"))

    if "title vi" in text or "language line" in text:
        sev = "Critical"
    elif "chest pain" in text or "er" in text or "emergency" in text:
        sev = "High"
    elif "sunday" in text or "closed" in text:
        sev = "High"
    elif sev == "Low":
        # Keep potential defects visible but avoid low/noise class.
        sev = "Medium"

    issue["severity"] = sev
    return issue


def _normalize_text(value):
    text = str(value or "").lower().strip()
    out = []
    prev_space = False
    for ch in text:
        if ch.isalnum():
            out.append(ch)
            prev_space = False
        else:
            if not prev_space:
                out.append(" ")
            prev_space = True
    return " ".join("".join(out).split())


def _dedupe_issues(issues):
    deduped = []
    seen_title_keys = set()
    seen_agent_keys = set()

    for issue in issues:
        source = str(issue.get("source_file", ""))
        category = _normalize_text(issue.get("category", ""))
        title = _normalize_text(issue.get("title", ""))[:120]
        agent = _normalize_text(issue.get("agent_said", ""))[:120]

        title_key = (source, category, title)
        agent_key = (source, category, agent)
        if title in ("", "untitled"):
            title_key = None
        if agent == "":
            agent_key = None

        if (title_key and title_key in seen_title_keys) or (agent_key and agent_key in seen_agent_keys):
            continue

        if title_key:
            seen_title_keys.add(title_key)
        if agent_key:
            seen_agent_keys.add(agent_key)
        deduped.append(issue)

    return deduped


def _cap_issues_per_call(issues, max_per_call=5):
    order = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3}
    grouped = {}
    for issue in issues:
        grouped.setdefault(issue.get("source_file", ""), []).append(issue)

    capped = []
    for _, group in grouped.items():
        group.sort(key=lambda x: order.get(x.get("severity", "Low"), 4))
        capped.extend(group[:max_per_call])
    return capped


# Topic families -- collapse multiple near-identical findings about the SAME topic
# within a single call down to one representative finding. This fixes the "10 passes
# each re-report the insurance gap with slightly different wording" duplication.
_TOPIC_PATTERNS = [
    ("insurance_network", ["in-network", "out-of-network", "out of network", "network status",
                            "copay", "co-pay", "insurance", "coverage", "billing"]),
    ("cancellation_policy", ["cancellation", "rescheduling", "reschedule policy", "notice window", "policy"]),
    ("multiple_medications", ["medication", "lisinopril", "atorvastatin", "refill", "two drugs", "drug"]),
    ("persona_rerouting", ["persona", "different person", "on behalf", "wrong person", "reroute", "rerouting"]),
    ("failed_transfer", ["transfer", "handoff", "hand-off", "representative", "test line", "disconnect", "connect you"]),
    ("wrong_number", ["wrong number", "cvs", "pharmacy", "wrong practice", "reached a different"]),
    ("language_barrier", ["hindi", "language line", "interpreter", "title vi", "language barrier", "language access"]),
    ("name_recognition", ["maria garcia", "name recognition", "full name", "verify", "verification", "look up", "lookup", "loop"]),
    ("directions_parking", ["parking", "directions", "garage", "lot", "entrance"]),
    ("follow_up_context", ["follow-up", "follow up", "returning patient", "already visited"]),
    ("triage_safety", ["chest", "triage", "er", "911", "emergency", "shortness of breath"]),
]


def _topic_key(issue):
    text = " ".join([
        str(issue.get("category", "")),
        str(issue.get("title", "")),
        str(issue.get("expected_behavior", "")),
    ]).lower()
    for topic, needles in _TOPIC_PATTERNS:
        if any(n in text for n in needles):
            return topic
    return None


def _collapse_by_topic(issues):
    # Within each call, keep only ONE finding per topic family (highest severity,
    # then the most detailed). Issues with no recognized topic pass through untouched.
    order = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3}

    def _detail_len(i):
        return len(str(i.get("expected_behavior", ""))) + len(str(i.get("impact", "")))

    best_by_key = {}
    passthrough = []
    for issue in issues:
        topic = _topic_key(issue)
        if topic is None:
            passthrough.append(issue)
            continue
        key = (issue.get("source_file", ""), topic)
        current = best_by_key.get(key)
        if current is None:
            best_by_key[key] = issue
            continue
        # Prefer higher severity, then more detail.
        cand_rank = (order.get(issue.get("severity", "Low"), 4), -_detail_len(issue))
        cur_rank = (order.get(current.get("severity", "Low"), 4), -_detail_len(current))
        if cand_rank < cur_rank:
            best_by_key[key] = issue

    return list(best_by_key.values()) + passthrough


_STOPWORDS = {
    "the", "a", "an", "and", "or", "to", "of", "for", "in", "on", "with", "did",
    "not", "no", "but", "agent", "caller", "call", "patient", "when", "that",
    "this", "was", "were", "is", "are", "then", "any", "provided", "provide",
}


def _title_tokens(issue):
    words = _normalize_text(issue.get("title", "")).split()
    return {w for w in words if w not in _STOPWORDS and len(w) > 2}


def _collapse_similar_titles(issues, threshold=0.5):
    # Final safety net: within each call, merge findings whose titles overlap
    # heavily (Jaccard >= threshold), keeping the highest-severity / most detailed.
    order = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3}

    def _detail_len(i):
        return len(str(i.get("expected_behavior", ""))) + len(str(i.get("impact", "")))

    def _better(a, b):
        ra = (order.get(a.get("severity", "Low"), 4), -_detail_len(a))
        rb = (order.get(b.get("severity", "Low"), 4), -_detail_len(b))
        return a if ra <= rb else b

    grouped = {}
    for issue in issues:
        grouped.setdefault(issue.get("source_file", ""), []).append(issue)

    result = []
    for _, group in grouped.items():
        kept = []
        kept_tokens = []
        for issue in group:
            toks = _title_tokens(issue)
            merged = False
            for idx, ktoks in enumerate(kept_tokens):
                union = toks | ktoks
                if not union:
                    continue
                jaccard = len(toks & ktoks) / len(union)
                if jaccard >= threshold:
                    kept[idx] = _better(kept[idx], issue)
                    kept_tokens[idx] = _title_tokens(kept[idx])
                    merged = True
                    break
            if not merged:
                kept.append(issue)
                kept_tokens.append(toks)
        result.extend(kept)
    return result


def _filter_and_reclassify(issues):
    # No reranking / no severity overrides -- keep the model's own severities,
    # including Low. Dedupe exact + collapse near-duplicates by topic and by
    # title similarity, then cap.
    normalized = [dict(i) for i in issues]
    for i in normalized:
        i["severity"] = _canonical_severity(i.get("severity"))
    deduped = _dedupe_issues(normalized)
    collapsed = _collapse_by_topic(deduped)
    collapsed = _collapse_similar_titles(collapsed)
    return _cap_issues_per_call(collapsed, max_per_call=5)


def analyze(transcript, filename):
    hindi_note = (
        "\n\nNOTE: Hindi scenario -- patient speaks only Hindi. "
        "[hi-IN]=Hindi, [en-US]=English.\n"
        if is_hindi(transcript) else ""
    )
    for attempt in range(1, 4):
        try:
            resp = client.chat.completions.create(
                model=CHAT_DEPLOYMENT,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": f"FILE: {filename}{hindi_note}\n\n{transcript}"},
                ],
                response_format={"type": "json_object"},
                reasoning_effort="low",
                max_completion_tokens=8000,
            )
        except Exception as e:
            print(f"     [ANALYZE ERROR] attempt {attempt}: {type(e).__name__}: {e}")
            continue

        choice = resp.choices[0]
        raw = (choice.message.content or "").strip()
        if not raw:
            # gpt-5-mini sometimes returns empty content (reasoning consumed the
            # token budget -> finish_reason='length', or a content filter fired).
            print(f"     [ANALYZE] {filename}: empty content "
                  f"(finish_reason={choice.finish_reason}); retry {attempt}/3")
            continue

        try:
            parsed = json.loads(raw)
            issues = parsed if isinstance(parsed, list) else next(
                (v for v in parsed.values() if isinstance(v, list)), []
            )
        except json.JSONDecodeError:
            print(f"     [ANALYZE] {filename}: non-JSON content; retry {attempt}/3")
            continue

        for issue in issues:
            issue["source_file"] = filename
            if is_hindi(transcript):
                issue["language_scenario"] = True
        return issues

    print(f"     [ANALYZE] {filename}: no usable response after 3 attempts")
    return []


def _call_number(source_file):
    name = str(source_file or "")
    parts = name.split("_")
    if len(parts) >= 2 and parts[1].isdigit() and len(parts[1]) == 2:
        return parts[1]
    return "unknown"


def build_report(issues):
    order = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3}
    issues.sort(key=lambda x: order.get(x.get("severity", "Low"), 4))
    counts = {s: sum(1 for i in issues if i.get("severity") == s)
              for s in ("Critical", "High", "Medium", "Low")}
    now   = datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")
    lang  = [i for i in issues if i.get("language_scenario")]
    groups = {}
    for issue in issues:
        cid = _call_number(issue.get("source_file", ""))
        groups.setdefault(cid, []).append(issue)
    lines = [
        "# Voice Agent Bug Report",
        f"Generated: {now}  |  {len(issues)} issues",
        "",
        "## Summary",
        "| Severity | Count |",
        "|----------|-------|",
        *[f"| {s} | {counts[s]} |" for s in ("Critical", "High", "Medium", "Low")],
        f"| **Total** | **{len(issues)}** |",
        "",
    ]
    if lang:
        lines += [
            "### Language Handling Issues",
            f"{len(lang)} issue(s) in Hindi scenario (10)",
            "",
            "> Note: Under Title VI, federally-funded medical practices must provide "
            "language access to limited-English-proficient patients.",
            "",
        ]
    lines += ["---", "", "## Issues Grouped By Call Number", ""]
    bug_index = 1
    for cid in sorted(groups.keys()):
        lines += [f"### Call {cid}", ""]
        call_issues = sorted(groups[cid], key=lambda x: order.get(x.get("severity", "Low"), 4))
        for i in call_issues:
            sev = i.get("severity", "?")
            flag = " [HINDI]" if i.get("language_scenario") else ""
            lines += [
                f"#### Bug #{bug_index:02d} -- {sev}: {i.get('title', 'Untitled')}{flag}",
                "",
                f"- **Category:** {i.get('category', '')}",
                f"- **Call:** {i.get('source_file', '?')} at {i.get('timestamp', '?')}",
                "",
                f"**Agent said:** {i.get('agent_said', '')}",
                "",
                f"**Expected:** {i.get('expected_behavior', '')}",
                "",
                f"**Impact:** {i.get('impact', '')}",
                "",
                "---",
                "",
            ]
            bug_index += 1
    return "\n".join(lines)


# Call-specific probes used during multi-pass analysis.
_probes = {
    "call_01": "FOCUSED PROBE for this call (new-patient scheduling): the agent booked a new patient. It MUST have collected insurance and stated copay + in-network/out-of-network status. If insurance/copay/network was never addressed, report HIGH (Missing Information / Insurance & Billing).",
    "call_02": "FOCUSED PROBE for this call (reschedule): the caller is rescheduling/cancelling. The agent MUST state the cancellation/rescheduling policy (notice window, fees). If it did not, report HIGH (Missing Information / Policy).",
    "call_03": "FOCUSED PROBE for this call (medication refill): the caller asked for TWO drugs -- lisinopril 10 mg AND atorvastatin 20 mg. Read the agent's final refill summary. If atorvastatin is missing / it says 'lisinopril 10 and 20 mg' / it collapsed two drugs into one drug at two strengths, report CRITICAL (Handling Error / Medication Safety). Quote the summary line.",
    "call_04": "FOCUSED PROBE for this call (insurance inquiry / persona): (a) if the agent set/confirmed insurance without ever stating in-network vs out-of-network, report HIGH (Insurance & Billing). (b) If the caller is acting for a different person than the registered patient and the agent kept collecting info without rerouting, report CRITICAL (Identity / Handling Error). (c) If the agent promised a transfer that dropped to a test line/goodbye, report HIGH (Handling Error).",
    "call_05": "FOCUSED PROBE for this call (follow-up patient): if the agent asked for or re-explained DIRECTIONS/address as if the caller were new, report MEDIUM (UX/Flow). Also flag any false 'we already did that' confirmation.",
    "call_06": "FOCUSED PROBE for this call (identity): the caller identifies as 'Maria Garcia' and the expected patient is 'Maria'. If the agent could not recognize they are the same person and looped on verification or aborted, report CRITICAL (Identity / NLU). If it then promised a transfer that dropped to a test line/goodbye, report HIGH (Handling Error).",
    "call_07": "FOCUSED PROBE for this call (urgent chest symptoms): if the agent failed to triage chest tightness / shortness of breath or did not mention ER/911, report HIGH (Missing Safety Check).",
    "call_08": "FOCUSED PROBE for this call (directions/parking): if the agent gave only general location guidance without specific PARKING details (garage/lot, entrance, validation), report MEDIUM (UX/Flow).",
    "call_09": "FOCUSED PROBE for this call (wrong number): the caller says they think they reached CVS Pharmacy. If the agent did NOT clearly correct them ('this is Pivot Point Orthopedics, not a pharmacy') and instead just collected/forwarded the prescription request, report CRITICAL (Handling Error / Routing).",
    "call_10": "FOCUSED PROBE for this call (Hindi language barrier): the caller speaks Hindi throughout (Devanagari, e.g. 'मेरा पूरा नाम मरिया गार्सिया है'). If the agent never acknowledged Hindi, never switched, and never offered an interpreter/language line, report CRITICAL (Language Handling) -- Title VI failure.",
}


def _probe_for(filename):
    for key, probe in _probes.items():
        if filename.startswith(key):
            return "\n\nCALL-SPECIFIC PROBE (apply to THIS transcript in addition to the general checks):\n" + probe
    return ""


def analyze_bugs_notebook(transcript=None, dry_run=False, n_passes=10, use_call_probes=True):
    global SYSTEM_PROMPT
    paths = [Path(transcript)] if transcript else sorted(TRANSCRIPTS_DIR.glob("*.txt"))
    if not paths:
        print(f"[!] No transcripts found in {TRANSCRIPTS_DIR}")
        return []

    print(f"Analyzing {len(paths)} transcript(s) via {CHAT_DEPLOYMENT} ({n_passes} pass(es)) ...\n")
    all_issues = []
    _orig_system_prompt = SYSTEM_PROMPT
    try:
        for path in paths:
            text = path.read_text(encoding="utf-8")
            hindi_tag = " [Hindi]" if is_hindi(text) else ""
            print(f"  -> {path.name}{hindi_tag}")
            if len(text.strip()) < 50:
                print("     (too short, skipping)")
                continue
            if dry_run:
                print("     (dry run)")
                continue

            SYSTEM_PROMPT = _orig_system_prompt + (_probe_for(path.name) if use_call_probes else "")

            per_call_issues = []
            for _ in range(max(1, int(n_passes))):
                per_call_issues.extend(analyze(text, path.name))

            per_call_issues = _dedupe_issues(per_call_issues)
            print(f"     unique issues after {n_passes} pass(es): {len(per_call_issues)}")
            all_issues.extend(per_call_issues)
    finally:
        SYSTEM_PROMPT = _orig_system_prompt

    if dry_run:
        return []

    filtered_issues = _filter_and_reclassify(all_issues)
    by_sev = {s: sum(1 for i in filtered_issues if i.get("severity") == s) for s in ("Critical", "High", "Medium", "Low")}
    OUTPUT_FILE.write_text(build_report(filtered_issues), encoding="utf-8")
    print(f"Report written: {OUTPUT_FILE}  ({len(filtered_issues)} issues -> Critical={by_sev['Critical']} High={by_sev['High']} Medium={by_sev['Medium']} Low={by_sev['Low']})")
    return filtered_issues


# Example usage in notebook:
# analyze_bugs_notebook(dry_run=True)
# analyze_bugs_notebook(n_passes=10)

[ANALYZER] env: /Users/asaraog/Documents/job/voicebot/runstoshare/.env
[ANALYZER] endpoint: https://asi5515-9030-resource.openai.azure.com
[ANALYZER] model: gpt-5-mini
[ANALYZER] transcripts dir: /Users/asaraog/Documents/job/voicebot/runstoshare/test_transcripts
[ANALYZER] report file: /Users/asaraog/Documents/job/voicebot/runstoshare/test_bug_report.md


# Main Orchestrator Functions

In [25]:
# main_orchestrator.py (notebook-friendly)
# Core orchestration functions:

# - run_calls_notebook(...): run selected scenarios and save recordings/transcripts
# - run_prod_and_bug_report(...): production run with tunnel/env sync and run-folder artifacts

import os, shutil, time, json, requests
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse

from dotenv import load_dotenv
BASE_DIR = Path.cwd().resolve()
load_dotenv(dotenv_path=BASE_DIR / ".env", override=True)

RECORDINGS_DIR  = BASE_DIR / "recordings";  RECORDINGS_DIR.mkdir(exist_ok=True)
TRANSCRIPTS_DIR = BASE_DIR / "transcripts"; TRANSCRIPTS_DIR.mkdir(exist_ok=True)
INTER_CALL_DELAY = 20


def run_bug_analysis():
    if "analyze_bugs_notebook" not in globals():
        print("[ANALYZE] Skipped: analyze_bugs_notebook is not loaded. Run the Bug Analyzer cell first.")
        return []
    print("\n" + "=" * 60)
    print("  AUTO-RUNNING BUG ANALYSIS")
    print("=" * 60 + "\n")
    return analyze_bugs_notebook(dry_run=False)


def run_calls_notebook(scenarios="all", dry_run=False, no_server=False, no_analyze=False):
    # Helpers are expected from Shared Utilities.
    required_helpers = [
        "start_server", "stop_server", "wait_for_server_ready",
        "make_call", "wait_for_completion", "download_recording", "transcribe",
    ]
    missing_helpers = [name for name in required_helpers if name not in globals()]
    if missing_helpers:
        raise RuntimeError(
            "Missing helper functions: " + ", ".join(missing_helpers) +
            ". Run the Shared Utilities and FastAPI Server cells first."
        )

    if "SCENARIOS" not in globals():
        raise RuntimeError("Missing SCENARIOS. Run the Scenarios cell first.")

    valid_ids = {s["id"] for s in SCENARIOS}
    if scenarios == "all":
        targets = SCENARIOS
    else:
        ids = [s.strip().zfill(2) for s in str(scenarios).split(",")]
        invalid = [i for i in ids if i not in valid_ids]
        if invalid:
            raise ValueError(f"Invalid IDs: {invalid}. Valid: {sorted(valid_ids)}")
        targets = [s for s in SCENARIOS if s["id"] in ids]

    print(f"\n{'-' * 60}")
    print(f"  VOICE BOT  {len(targets)} scenario(s)  -> +18054398008")
    print(f"{'-' * 60}\n")

    if dry_run:
        for s in targets:
            print(f"  [{s['id']}] {s['name']}  |  {s['description']}")
        return []

    server = None
    if not no_server:
        server = start_server()
        wait_for_server_ready()

    results = []
    try:
        for i, scenario in enumerate(targets, 1):
            lang = " [Hindi]" if scenario["id"] == "10" else ""
            print(f"\n[{i}/{len(targets)}] Scenario {scenario['id']}: {scenario['name']}{lang}")
            result = {"id": scenario["id"], "name": scenario["name"], "status": "error"}
            try:
                call_sid = make_call(scenario)
                call     = wait_for_completion(call_sid)
                duration = int(call.duration or 0)
                result.update({"call_sid": call_sid, "status": call.status, "duration": duration})
                if call.status == "completed" and duration >= 4:
                    rec = download_recording(
                        call_sid,
                        RECORDINGS_DIR / f"call_{scenario['id']}_{call_sid[:8]}.mp3"
                    )
                    if rec:
                        transcribe(
                            rec, scenario["id"],
                            TRANSCRIPTS_DIR / f"call_{scenario['id']}_{call_sid[:8]}_transcript.txt"
                        )
            except Exception as e:
                print(f"     [ERROR] {e}")
                result["error"] = str(e)
            results.append(result)
            if i < len(targets):
                print(f"     next call in {INTER_CALL_DELAY}s ...")
                time.sleep(INTER_CALL_DELAY)
    finally:
        stop_server(server)

    done = [r for r in results if r.get("status") == "completed"]
    print(f"\n  DONE  {len(done)}/{len(results)} completed")
    Path("call_results.json").write_text(
        json.dumps({"ts": datetime.utcnow().isoformat(), "results": results}, indent=2)
    )
    if not no_analyze:
        run_bug_analysis()
    return results


def run_prod_and_bug_report(scenarios="all"):
    ARTIFACT_BASE = Path.cwd().resolve()
    ENV_FILE = ARTIFACT_BASE / ".env"
    load_dotenv(dotenv_path=ENV_FILE, override=True)

    def _write_env_key(env_path, key, value):
        lines = env_path.read_text(encoding="utf-8").splitlines() if env_path.exists() else []
        new_line = f"{key}={value}"
        replaced = False
        out = []
        for line in lines:
            if line.strip().startswith(f"{key}="):
                out.append(new_line)
                replaced = True
            else:
                out.append(line)
        if not replaced:
            out.append(new_line)
        env_path.write_text("\n".join(out) + "\n", encoding="utf-8")

    def _discover_tunnel_host():
        for key in ("LOCALTUNNEL_URL", "TUNNEL_URL", "PUBLIC_BASE_URL", "SERVER_DOMAIN"):
            raw = os.environ.get(key, "").strip()
            if not raw:
                continue
            host = urlparse(raw).netloc if raw.startswith("http") else raw
            if host and "." in host:
                return host, key
        return "", ""

    def _validate_public_health(domain, max_checks=8, delay_sec=2):
        url = f"https://{domain}/health"
        last_err = ""
        for attempt in range(1, max_checks + 1):
            try:
                r = requests.get(url, timeout=8)
                if r.status_code == 200:
                    print(f"[PUBLIC] Ready: {url}")
                    return
                last_err = f"status={r.status_code} body={r.text[:120]}"
            except Exception as e:
                last_err = f"{type(e).__name__}: {str(e)[:120]}"
            print(f"[PUBLIC] waiting ({attempt}/{max_checks}) ... {last_err}")
            time.sleep(delay_sec)
        raise RuntimeError(f"Public webhook not reachable at {url}. Last error: {last_err}")

    required = ["start_server", "stop_server", "wait_for_server_ready", "run_calls_notebook", "analyze", "_filter_and_reclassify", "build_report"]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            "Missing helpers: " + ", ".join(missing) +
            ". Run Shared Utilities, FastAPI Server, Main Orchestrator, and Bug Analyzer cells first."
        )

    # Resolve target + tunnel and persist to .env
    target_number = os.environ.get("TARGET_NUMBER", "").strip()
    if not target_number:
        raise RuntimeError("TARGET_NUMBER not set in .env")

    domain, source_key = _discover_tunnel_host()
    if not domain:
        raise RuntimeError("No tunnel host found. Set LOCALTUNNEL_URL (or SERVER_DOMAIN) in .env")

    os.environ["TARGET_NUMBER"] = target_number
    os.environ["SERVER_DOMAIN"] = domain
    globals()["SERVER_DOMAIN"] = domain
    _write_env_key(ENV_FILE, "TARGET_NUMBER", target_number)
    _write_env_key(ENV_FILE, "SERVER_DOMAIN", domain)
    if not os.environ.get("LOCALTUNNEL_URL", "").strip():
        _write_env_key(ENV_FILE, "LOCALTUNNEL_URL", f"https://{domain}")

    print("Calling TARGET_NUMBER:", target_number)
    print(f"Tunnel host: {domain} (source={source_key})")
    print(f"Synced SERVER_DOMAIN in {ENV_FILE}")

    recordings_dir = ARTIFACT_BASE / "recordings"
    transcripts_dir = ARTIFACT_BASE / "transcripts"
    recordings_dir.mkdir(exist_ok=True)
    transcripts_dir.mkdir(exist_ok=True)

    # Clean server restart to avoid stale notebook server state.
    try:
        if "stop_notebook_server" in globals():
            stop_notebook_server()
    except Exception:
        pass
    if "_uvicorn_server" in globals():
        globals()["_uvicorn_server"] = None
    if "_uvicorn_thread" in globals():
        globals()["_uvicorn_thread"] = None
    time.sleep(1.0)

    run_started_at = time.time()
    server = None
    try:
        server = start_server(target_number=target_number)
        wait_for_server_ready(max_wait=25)
        _validate_public_health(domain)

        run_output_target = run_calls_notebook(
            scenarios=scenarios, dry_run=False, no_server=True, no_analyze=True
        )

        # Create the run folder up front so the bug report writes straight into runs/target_<stamp>/.
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        run_dir = ARTIFACT_BASE / "runs" / f"target_{stamp}"
        (run_dir / "recordings").mkdir(parents=True, exist_ok=True)
        (run_dir / "transcripts").mkdir(parents=True, exist_ok=True)
        (run_dir / "vad_timelines").mkdir(parents=True, exist_ok=True)

        # Build transcript set from THIS run only.
        this_run_paths = []
        for r in run_output_target:
            sid8 = str(r.get("call_sid", ""))[:8]
            sid = str(r.get("id", "")).zfill(2)
            if not sid8:
                continue
            p = transcripts_dir / f"call_{sid}_{sid8}_transcript.txt"
            if p.exists():
                this_run_paths.append(p)

        paths_for_report = sorted(this_run_paths, key=lambda x: x.name)
        print("Transcripts in report:", len(paths_for_report))
        for p in paths_for_report:
            print(" -", p.name)

        # Analyze just the selected transcripts; report goes directly into the run folder only.
        global TRANSCRIPTS_DIR, OUTPUT_FILE
        TRANSCRIPTS_DIR = transcripts_dir
        OUTPUT_FILE = run_dir / "bug_report.md"
        all_issues = []
        for p in paths_for_report:
            text = p.read_text(encoding="utf-8")
            if len(text.strip()) < 50:
                continue
            all_issues.extend(analyze(text, p.name))
        issues_target = _filter_and_reclassify(all_issues)
        OUTPUT_FILE.write_text(build_report(issues_target), encoding="utf-8")
        print(f"Report written: {OUTPUT_FILE}  ({len(issues_target)} issues)")
    finally:
        stop_server(server)

    # Archive this run's recordings + transcripts + VAD timelines into the run folder.
    for p in recordings_dir.glob("*.mp3"):
        if p.stat().st_mtime >= run_started_at - 2:
            shutil.copy2(p, run_dir / "recordings" / p.name)

    # Diarized transcripts (*_transcript.txt).
    for p in transcripts_dir.glob("*_transcript.txt"):
        if p.stat().st_mtime >= run_started_at - 2:
            shutil.copy2(p, run_dir / "transcripts" / p.name)

    # Raw VAD timeline files (call_XX_<timestamp>.txt, written by the server).
    for p in transcripts_dir.glob("call_[0-9][0-9]_20*_*.txt"):
        if p.stat().st_mtime >= run_started_at - 2:
            shutil.move(str(p), str(run_dir / "vad_timelines" / p.name))

    call_results = ARTIFACT_BASE / "call_results.json"
    if call_results.exists():
        shutil.copy2(call_results, run_dir / "call_results.json")

    print("\nCompleted scenarios:", len(run_output_target))
    print("Total issues:", len(issues_target))
    print("Bug report:", OUTPUT_FILE.resolve())
    print("Run folder:", run_dir.resolve())
    return {
        "results": run_output_target,
        "issues": issues_target,
        "run_dir": str(run_dir),
        "report": str(OUTPUT_FILE.resolve()),
    }


# Example usage in notebook:
# run_calls_notebook(scenarios="01", dry_run=True)
# run_prod_and_bug_report(scenarios="all")

# Single Command Run

Runs all 10 scenarios against your test number, downloads recordings, transcribes, and generates a bug report.

> **Pre-flight:** start localtunnel, set `LOCALTUNNEL_URL` in `.env`, then run this cell.

```bash
npx localtunnel --port 8080   # keep running in a separate terminal
```

In [26]:
# Run scenario 7 by itself.
run_output = run_prod_and_bug_report(scenarios="07")
print("\nCompleted scenarios:", len(run_output["results"]))
print("Total issues:", len(run_output["issues"]))
print("Bug report path:", run_output["report"])
print("Run folder:", run_output["run_dir"])

Calling TARGET_NUMBER: +18054398008
Tunnel host: wicked-buttons-tan.loca.lt (source=LOCALTUNNEL_URL)
Synced SERVER_DOMAIN in /Users/asaraog/Documents/job/voicebot/runstoshare/.env
Server is not running.


INFO:     Started server process [40545]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8080 (Press CTRL+C to quit)


Server started on http://127.0.0.1:8080
INFO:     127.0.0.1:63304 - "GET /health HTTP/1.1" 200 OK
[SERVER] Ready
INFO:     108.204.1.52:0 - "GET /health HTTP/1.1" 200 OK
[PUBLIC] Ready: https://wicked-buttons-tan.loca.lt/health

------------------------------------------------------------
  VOICE BOT  1 scenario(s)  -> +18054398008
------------------------------------------------------------


[1/1] Scenario 07: urgent_same_day
INFO:     127.0.0.1:63306 - "POST /make-call HTTP/1.1" 200 OK
  SID: CAda7d32dd6eb99e070bb372ea6c9fd503
INFO:     184.72.80.226:0 - "POST /call-handler HTTP/1.1" 200 OK


INFO:     13.217.89.103:0 - "WebSocket /media-stream" [accepted]


[BRIDGE] opening azure websocket: wss://asi5515-9030-resource.openai.azure.com/openai/v1/realtime?model=gpt-realtime-mini


...

[BRIDGE] azure websocket connected
[AZURE] detected_output_codec=pcm16_24k


INFO:     connection closed


     status: completed
INFO:     35.171.17.66:0 - "POST /recording-status HTTP/1.1" 200 OK
     recording -> /Users/asaraog/Documents/job/voicebot/runstoshare/recordings/call_07_CAda7d32.mp3


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [40545]


     transcript -> /Users/asaraog/Documents/job/voicebot/runstoshare/transcripts/call_07_CAda7d32_transcript.txt  (32 phrases)
Server stop requested.

  DONE  1/1 completed
Transcripts in report: 1
 - call_07_CAda7d32_transcript.txt
Report written: /Users/asaraog/Documents/job/voicebot/runstoshare/runs/target_20260701_144650/bug_report.md  (0 issues)
Server stop requested.

Completed scenarios: 1
Total issues: 0
Bug report: /Users/asaraog/Documents/job/voicebot/runstoshare/runs/target_20260701_144650/bug_report.md
Run folder: /Users/asaraog/Documents/job/voicebot/runstoshare/runs/target_20260701_144650

Completed scenarios: 1
Total issues: 0
Bug report path: /Users/asaraog/Documents/job/voicebot/runstoshare/runs/target_20260701_144650/bug_report.md
Run folder: /Users/asaraog/Documents/job/voicebot/runstoshare/runs/target_20260701_144650
